# TC-WPN — Phase 1: baseline-collapse diagnosis + auxiliary-head isolation

**Accelerator: GPU T4.** This is your supervisor's Phase 1, items 1–3, in the
order they were specified:

1. Debug ProtoNet's 0.5 collapse — *first*
2. Auxiliary-head isolation — *scientifically mandatory*
3. Inspect prototype separation and gradients

Budget: two training runs (~95 min) plus diagnostics on the five existing
Stage C checkpoints (inference only, a few minutes). No seed sweep, no K sweep,
no tuning — those are Phase 2 and 3, and your supervisor was explicit that
running them now would be premature.

## Why the collapse matters more than the AUROC

Four of five Stage C models emitted a constant 0.5:

| run | p_sd | mean p (case) | mean p (control) | AUROC |
|---|---:|---:|---:|---:|
| protonet | 0.0001 | 0.5000 | 0.5000 | 0.522 |
| protonet_temp | 0.0005 | 0.5000 | 0.5000 | 0.499 |
| pcw_only | 0.0005 | 0.5000 | 0.5000 | 0.503 |
| temporal_only | 0.0013 | 0.5000 | 0.4999 | 0.510 |
| **tcwpn_full** | **0.2149** | **0.5822** | **0.4053** | **0.738** |

The `temporal_only` manifest shows the episodic loss at 0.660–0.736 across all
3,000 steps. ln(2) = 0.693. It never moved — while `tau` learned 10.0 → 6.53 and
`lambda` 0.500 → 0.400, so the optimiser was working.

Your supervisor's framing is the right one: until we know *why* the baseline
collapses, the ΔAUROC = 0.216 at p ≈ 2e-45 cannot be attributed to anything.

## The two hypotheses

`tcwpn_full` is the only preset with `aux_head_weight > 0`. Two runs separate
the explanations:

- **`temporal_pcw`** — both mechanisms, no auxiliary head *(config existed;
  Stage C never launched it)*
- **`aux_only`** — auxiliary head, neither mechanism *(added for this run;
  this is your supervisor's configuration B/E)*

| outcome | conclusion |
|---|---|
| `aux_only` ≈ 0.73, `temporal_pcw` ≈ 0.50 | the auxiliary loss does the work; w^T and w^C add nothing |
| `aux_only` < `tcwpn_full`, both > 0.50 | interaction effect — supervisor's item 31, the most interesting outcome |
| `temporal_pcw` ≈ 0.73 | stop and re-read the training code |

Write your prediction down before running.

In [ ]:
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!pip install -q -r requirements.txt 2>&1 | tail -2

import subprocess, sys, os
r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/test_repo_layout.py",   # missing modules, stray duplicates
                    "tests/test_call_arity.py",    # wrong-arg-count calls
                    "-q", "--no-header"],
                   capture_output=True, text=True,
                   env={**os.environ, "PYTHONPATH": "src"})
print(r.stdout[-2000:])
if r.returncode != 0:
    raise SystemExit("Repository layout is broken — fix before spending GPU time.")

In [ ]:
from pathlib import Path
STAGE_A_DS = Path("/kaggle/input/datasets/dulharakaushalya/tc-wpn-stage-a-data")
STAGE_A = next((c for c in (STAGE_A_DS/"data"/"clean", STAGE_A_DS/"clean", STAGE_A_DS)
                if (c/"pkl").exists()), None)
if STAGE_A is None:
    raise SystemExit(f"no pkl/ under {STAGE_A_DS}")

PKL_DIR, PLAN_DIR = "/kaggle/working/pkl", str(STAGE_A/"plans")
STEM, K, SEED, RESULTS = "psych_mimic4idx", 5, 42, "/kaggle/working/results"
!mkdir -p {PKL_DIR}
!cp {STAGE_A}/pkl/*.pkl {PKL_DIR}/
print("ready")

## Collapse diagnostics — supervisor items 3 and 33

`diagnose_collapse.py` runs inference over validation episodes on each existing
checkpoint and reports exactly what was asked for:

- `cos(p_case, p_control)` — prototype separation
- mean pairwise cosine within support — embedding variance
- `d(query, control proto) - d(query, case proto)` — the distance gap, split by
  true label
- `tau`
- gradient L2 norm by parameter group — encoder / projection / temperature /
  weight modules / aux head

It also checks the distance gap's **sign**. Both `protonet` and `protonet_temp`
landed slightly *below* 0.5, which is what an inverted class-to-column mapping
would produce, so that is worth ruling out explicitly rather than assuming.

This is cheap — no training — so run it on all five Stage C checkpoints.

In [ ]:
# ---------------------------------------------------------------------------
# Locate the Stage C checkpoints. Auto-discovered: any best.pt under
# /kaggle/input or in this session's results dir. No placeholder to edit.
# ---------------------------------------------------------------------------
import os, glob

CONFIGS_TO_DIAGNOSE = ["protonet", "protonet_temp", "temporal_only",
                       "pcw_only", "tcwpn_full"]

def find_checkpoint(cfg):
    """Return the run dir containing best.pt for this config, or None."""
    wanted = f"{cfg}_k{K}_seed{SEED}"
    local = f"{RESULTS}/{STEM}/{wanted}"
    if os.path.exists(f"{local}/best.pt"):
        return local
    hits = [os.path.dirname(p)
            for p in glob.glob(f"/kaggle/input/**/{wanted}/best.pt", recursive=True)]
    return sorted(hits)[0] if hits else None

found, missing = {}, []
for cfg in CONFIGS_TO_DIAGNOSE:
    run = find_checkpoint(cfg)
    (found.__setitem__(cfg, run) if run else missing.append(cfg))

print(f"checkpoints found: {sorted(found)}")
if missing:
    print(f"checkpoints MISSING: {missing}")
    print("\nAdd your Stage C results as an input dataset (+ Add Input).")
    print("Any best.pt under /kaggle/input is discovered automatically.")
if not found:
    print("\nNo checkpoints available. The diagnostic cells below will be")
    print("skipped; the training cells further down still work and will")
    print("produce checkpoints you can diagnose on a second pass.")

for cfg, run in found.items():
    print("=" * 70)
    print(f"DIAGNOSING {cfg}  ({run})")
    !python -m scripts.diagnose_collapse --run {run} \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} \
        --split val --episodes 100 --grad-episodes 20 \
        --out /kaggle/working/collapse_{cfg}.json

In [ ]:
# ---------------------------------------------------------------------------
# Side-by-side: the collapsed models against the one that learned.
# ---------------------------------------------------------------------------
import json, glob, os
import pandas as pd

files = sorted(glob.glob("/kaggle/working/collapse_*.json"))
if not files:
    print("No diagnostic output found in /kaggle/working/.")
    print("The cell above found no checkpoints, or every diagnostic run failed.")
    print("Scroll up and read the FIRST error, not the last.")
else:
    rows = []
    for f in files:
        d = json.load(open(f))
        g, gn = d["geometry"], d["grad_norms"]
        pick = lambda k: round(g[k]["mean"], 5) if g.get(k) else None
        rows.append({
            "run": d["run"],
            "proto_cos": pick("proto_cos"),
            "support_spread": pick("support_spread_all"),
            "gap_case": pick("gap_case"),
            "gap_control": pick("gap_control"),
            "gap_abs": pick("gap_abs_mean"),
            "logit_spread": pick("logit_spread"),
            "p_sd": pick("p_pos_sd"),
            "tau": pick("tau"),
            "grad_encoder": f"{gn.get('encoder', 0):.2e}",
            "grad_projection": f"{gn.get('projection', 0):.2e}",
        })
    df = pd.DataFrame(rows).set_index("run")
    print(df.to_string())
    df.to_csv("/kaggle/working/collapse_comparison.csv")
    print()
    print("The comparison that matters: whatever differs between the collapsed")
    print("runs and tcwpn_full IS the mechanism. If proto_cos ~ 1.0 for the four")
    print("and clearly below for tcwpn_full, the auxiliary loss is what keeps the")
    print("representation from collapsing -- and that is the paper's finding.")

In [ ]:
# Two runs, ~48 min each on a T4.
for cfg in ["aux_only", "temporal_pcw"]:
    print("="*70, f"\nTRAINING {cfg}\n", "="*70, sep="")
    !python -m scripts.train --config configs/{cfg}.yaml \
        --k {K} --seed {SEED} --stem {STEM} \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --results {RESULTS}

In [ ]:
for cfg in ["aux_only", "temporal_pcw"]:
    run = f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{SEED}"
    !python -m scripts.evaluate --run {run} --split test \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000

## The verdict

Two diagnostics decide it. **The loss curve matters more than the AUROC** — a
loss pinned at 0.693 means the model never learned, whichever way the AUROC
happened to land.

In [ ]:
import json, glob, os
import pandas as pd

rows = []
for run in sorted(glob.glob(f"{RESULTS}/{STEM}/*")):
    mf, ev = os.path.join(run, "manifest.json"), os.path.join(run, "eval_test.json")
    if not os.path.exists(mf):
        continue
    m = json.load(open(mf))
    h = m.get("history", [])
    rec = {
        "run": os.path.basename(run),
        "aux": m["config"]["model"].get("preset", "?"),
        "loss_first": round(h[0]["loss"], 4) if h else None,
        "loss_last": round(h[-1]["loss"], 4) if h else None,
        "loss_min": round(min(e["loss"] for e in h), 4) if h else None,
        "best_val": round(m.get("best_val_auroc_quick", float("nan")), 4),
        "tau_final": round(h[-1]["tau"], 3) if h else None,
        "lambda_final": (round(h[-1]["lambda_decay"], 3)
                         if h and h[-1].get("lambda_decay") else None),
        "beta_final": (round(h[-1]["beta_consistency"], 3)
                       if h and h[-1].get("beta_consistency") else None),
    }
    if os.path.exists(ev):
        rec["test_auroc"] = round(json.load(open(ev))["metrics"]["auroc"], 4)
    rows.append(rec)

df = pd.DataFrame(rows).set_index("run")
print(df.to_string())
print()
print("ln(2) = 0.6931. A loss_min at or above ~0.68 means the episodic")
print("objective never separated the classes, regardless of AUROC.")
df.to_csv("/kaggle/working/stage_d_training_diagnostics.csv")

In [ ]:
# Score spread: the difference between a weak model and an untrained one.
rows = []
for f in sorted(glob.glob(f"{RESULTS}/{STEM}/*/predictions_test.csv")):
    d = pd.read_csv(f); p = d["p_anxiety"]
    rows.append({"run": os.path.basename(os.path.dirname(f)),
                 "p_sd": round(p.std(), 4),
                 "mean_p_case": round(p[d.label == 1].mean(), 4),
                 "mean_p_control": round(p[d.label == 0].mean(), 4),
                 "separation": round(p[d.label == 1].mean() - p[d.label == 0].mean(), 4)})
sd = pd.DataFrame(rows).set_index("run")
print(sd.to_string())
sd.to_csv("/kaggle/working/stage_d_score_spread.csv")
print()
print("separation < 0.01 with p_sd < 0.01  ->  untrained, not weak.")

In [ ]:
import subprocess, json
PAIRS = [("aux_only", "tcwpn_full", "do w^T and w^C add anything over the aux head?"),
         ("temporal_pcw", "tcwpn_full", "does the aux head add anything over the mechanisms?"),
         ("protonet_temp", "aux_only", "does the aux head alone rescue training?")]
out = []
for a, b, why in PAIRS:
    pa = f"{RESULTS}/{STEM}/{a}_k{K}_seed{SEED}/predictions_test.csv"
    pb = f"{RESULTS}/{STEM}/{b}_k{K}_seed{SEED}/predictions_test.csv"
    dst = f"/kaggle/working/delong_{a}_vs_{b}.json"
    subprocess.run(["python", "-m", "scripts.compare_models", "pair",
                    "--a", pa, "--b", pb, "--out", dst], check=False)
    try:
        d = json.load(open(dst)); d["question"] = why; out.append(d)
    except Exception as e:
        print("skipped", a, b, e)
if out:
    t = pd.DataFrame(out)[["question", "auroc_a", "auroc_b", "delta_auroc", "p_value"]]
    print(t.to_string(index=False))
    t.to_csv("/kaggle/working/stage_d_delong.csv", index=False)

## What to tell your supervisor

If the expected pattern holds, the paper's claim changes and gets **more**
interesting, not less:

> Episodic prototypical training alone failed to train a clinical encoder in
> this regime — five configurations held the episodic loss at ln(2) across
> 3,000 episodes and produced constant predictions. Adding a lightweight
> per-note supervised auxiliary loss (weight 0.3) was necessary and sufficient
> for the encoder to learn; the temporal and prototype-consistency weightings
> contributed nothing measurable on top of it.

That is a negative result about few-shot clinical NLP with a concrete,
reproducible cause, on a benchmark you already showed is free of label leakage
and patient leakage. It is publishable, and it is honest.

**Do not report Stage C's +0.239 as evidence for TC-WPN.** Every comparison in
that ladder was against a model that never trained.

Two things not to skip:

1. **Report the blinding result as it stands.** TC-WPN's above-chance margin
   falls from 0.238 to 0.128 under `anx_meds` — roughly 46% of the signal was
   lexical. Also note that `anxiety` and `anx_meds` blinding gave nearly
   identical results (0.6284 vs 0.6291), so the drug names carried no
   additional signal beyond the diagnosis terms.
2. **The operating point is poor and needs saying.** At the locked threshold
   `tcwpn_full` runs at sensitivity 0.922, specificity 0.285. Threshold
   selection used `f1`, which at 59.4% prevalence pushes hard toward predicting
   positive. Consider reporting Youden-J as well, and never quote F1 alone: an
   all-positive classifier scores 0.745 on this cohort.

## Appendix — Phase 2 preparation only, do not run today

Supervisor items 25–26: the tokenizer keeps only the first 512 tokens, and
anxiety evidence may appear later in a discharge summary. Multi-chunk needs
**no code change** — `--max-chunks` already exists and the embedder mean-pools
chunks. And `store_fingerprint` hashes record count plus note IDs only, so
2- and 4-chunk pkls remain compatible with the frozen episode plans.

Two constraints before running this:

1. **Validation only.** Choose the chunk count on val, freeze it, then touch
   test once. Your supervisor was explicit (items 24, 34).
2. **Phase 1 first.** If the collapse turns out to be a representation problem,
   4 chunks of a collapsed representation is still collapsed. Diagnose, then
   improve.

Cost also rises roughly linearly with chunk count: 4 chunks is ~4× the forward
passes, so a 48-minute run becomes roughly three hours. Budget the quota.

In [ ]:
# PHASE 2 — leave commented until Phase 1 is read.
# COHORT = str(STAGE_A / "cohort_psych_mimic4idx.csv")
# for nc in [2, 4]:
#     !python -m scripts.tokenize_cohort --cohort {COHORT} \
#         --out /kaggle/working/pkl_chunk{nc} --blind none --max-chunks {nc}